# GraphGuard: 03 - Leakage-Safe Feature Engineering
**Objective**: Engineer chronological velocity, historical customer deviations, entity reuse metrics, and fit transformation pipeline strictly on the training partition (FR-03).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.feature_engineering import prepare_feature_data

# Execute feature pipeline
feat_dict = prepare_feature_data(
    input_csv=os.path.join("..", "data", "processed", "transactions_cleaned.csv"),
    save_pipeline_path=os.path.join("..", "models", "feature_pipeline.joblib"),
    output_processed_dir=os.path.join("..", "data", "processed")
)

print(f"Features Engineered ({len(feat_dict['feature_names'])}):")
for f in feat_dict["feature_names"]:
    print(f" - {f}")

### Train / Validation / Test Chronological Partition Validation

In [ ]:
train_df = feat_dict["train_df"]
val_df = feat_dict["val_df"]
test_df = feat_dict["test_df"]

print(f"Train Partition: {len(train_df)} samples ({train_df['is_fraud'].sum()} frauds) | Max Date: {train_df['timestamp'].max()}")
print(f"Val Partition:   {len(val_df)} samples ({val_df['is_fraud'].sum()} frauds) | Date Range: {val_df['timestamp'].min()} to {val_df['timestamp'].max()}")
print(f"Test Partition:  {len(test_df)} samples ({test_df['is_fraud'].sum()} frauds) | Date Range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")

assert train_df['timestamp'].max() <= val_df['timestamp'].min(), "Temporal leakage detected between train and val!"
assert val_df['timestamp'].max() <= test_df['timestamp'].min(), "Temporal leakage detected between val and test!"
print("Zero temporal look-ahead leakage verified successfully!")